# The Benefit of Data Sharing
Godwin et al. (2025) reviewed the open-science practices in recent **visual search** literature.<br>
They gracefully provided the dataset of articles they reviewed ([link](https://osf.io/5tmey/overview)), allowing others to explore the data further.<br>
Here, we analyze their dataset to examine whether sharing data is associated with increased citations. We rely on Godwin et al.'s classification of articles into four categories based on their data-sharing practices:
1. No data shared
2. Per-subject data
3. Per-trial data
4. Per-fixation data

We use the [OpenAlex](https://openalex.org/) API to retrieve citation counts for each of their articles and additional features of the publications (e.g. whether they were published with open access). We use these features, together with articles' data-sharing class, to predict their citations counts and [FWCI](https://help.openalex.org/hc/en-us/articles/24735753007895-Field-Weighted-Citation-Impact-FWCI) scores, and assess whether sharing data is associated with increased citations.

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [ ]:
import sys
from pathlib import Path

# notebooks live in `analysis/`, which must be importable for `helpers` to resolve
_ANALYSIS_DIR = Path.cwd() if (Path.cwd() / "helpers").is_dir() else Path.cwd() / "analysis"
if str(_ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(_ANALYSIS_DIR))

from typing import Literal, Dict
from itertools import combinations
from copy import deepcopy

import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers import dataset
from helpers.config import (
    CLASS_COLORS, SHARING_CLASS_ORDER, BINARY_FEATURES,
    TITLE_FONT, AXIS_TITLE_FONT, AXIS_TICK_FONT, LEGEND_FONT, FONT_FAMILY,
    VENUE_IMPACT_METRIC, VENUE_IMAPCT_METRIC_NAME,
)
from helpers.plotting import save_figure

pio.renderers.default = "browser"

combined, FEATURES_DF, CITATIONS_DF = dataset.load_or_build()

### Apply Exclusion Criteria
The Godwin et al. (2025) dataset contains over 500 records (of different articles). Our analysis focuses on a subset of articles that meet the following criteria:
1. A journal article - not a standalone OSF repo/project
2. Human eye-tracking data
3. Visual search related
4. Published within the Godwin et al. (2025) review period of 2017-2022 (though see caveat for 2016 articles below)
5. Has a valid DOI based on our OpenAlex query
6. Not retraced (based on the Godwin records)
7. not a duplicate of a different record (based on title-author or DOI match)

### Finalize Analytical Dataset
We combine the Godwin et al. (2025) data with the metadata we retrieved from OpenAlex to create our final analytical dataset.<br>
As a convension, `ALL_CAPS` column names are from the Godwin dataset; `CamelCase` column names are from OpenAlex metadata; and `snake_case` column names are derived or computed in this notebook.

In [ ]:
FINAL_DATASET_SIZE = len(combined)
combined.columns

---

## Check Discrepancies
We examine some discrepancies between the Godwin et al. (2025) dataset and the OpenAlex metadata we retrieved.

### Publication-Year Discrepancies
Some records in the Godwin dataset have a different publication year than in OpenAlex metadata.<br>
We show here how many discrepancies there are, but do not exclude any articles based on this.

In [ ]:
# the pre-exclusion frame, needed to see what each criterion removes
merged = dataset.build_merged()

In [ ]:
# exclude publications but disregard publication year criterion:
merged_subset = (
    merged
    .loc[  # correct topic
        merged[["IS_PRIMARY_RESEARCH_HUMAN", "IS_VISUAL_SEARCH", "IS_EYE_TRACKING"]].eq("YES").all(axis=1)
    ]
    .loc[merged["DOI"].notna()]  # successful query
    .loc[merged["IsRetracted"] == False]  # not retracted
    .loc[~merged.duplicated(subset="DOI", keep="first")]  # not dup DOI
)
print(f"Num of publications without excluding by publication year:\t {merged_subset.shape[0]}")

# calculate how many articles would be rejected by either database's publication year:
godwin_pass_year_range = merged_subset["YEAR_PUBLISHED"].between(2017, 2022).rename("Godwin")
openalex_pass_year_range = merged_subset["PublicationYear"].between(2017, 2022).rename("OpenAlex")
confusion = pd.crosstab(
    godwin_pass_year_range, openalex_pass_year_range, rownames=["Godwin"], colnames=["OpenAlex"],
)
display(confusion)

print("Publication-year discrepancy:")
display(
    merged_subset.loc[
        godwin_pass_year_range & ~openalex_pass_year_range, ["YEAR_PUBLISHED", "PublicationYear"]
    ].rename(columns={"YEAR_PUBLISHED": "Godwin_Year", "PublicationYear": "OpenAlex_Year"})
)

### Author Discrepancies

In [ ]:
print("### Missing Authorship ###")
godwin_missing_auths_count = combined["AUTHORS"].isna().sum()
print(f"Articles with missing authorship in the Godwin Dataset:\t {godwin_missing_auths_count} ({100 * godwin_missing_auths_count / FINAL_DATASET_SIZE :.1f}%)")
openalex_missing_auths_count = combined["Authors"].map(lambda auths: not auths or auths == "[]").sum()
print(f"Articles with missing authorship in the OpenAlex Dataset:\t {openalex_missing_auths_count} ({100 * openalex_missing_auths_count / FINAL_DATASET_SIZE :.1f}%)")

print ("### Authorship Discrepancy ###")
has_godwin_authors = (
    combined
    .loc[combined["AUTHORS"].notna(), ["DOI", "AUTHORS", "Authors", "NumAuthors"]]
    .rename(columns={
        "AUTHORS": "AUTHORS (Godwin)", "Authors": "Authors (OpenAlex)", "NumAuthors": "NumAuthors (OpenAlex)"
    })
)
has_godwin_authors["NUM_AUTHORS (Godwin)"] = has_godwin_authors["AUTHORS (Godwin)"].str.split("; ").str.len()
is_auth_count_bad = has_godwin_authors["NUM_AUTHORS (Godwin)"] != has_godwin_authors["NumAuthors (OpenAlex)"]
print(f"There are {is_auth_count_bad.sum()} ({100 * is_auth_count_bad.mean() :.1f}%) rows with mismatching author-counts")
has_godwin_authors.loc[is_auth_count_bad]

---

## Descriptive Statistics
Before diving into the analysis of impact of data-sharing on citations, we examine the features of the articles in our dataset, their distributions and the correlations between them.<br>
NOTE: **We do not calculate citation-statistics in this section**, to avoid biasing our analysis (double-dipping). We will calculate citation statistics in the next section, after we have defined our analytical dataset.

### Data Sharing Class
The Godwin et al. (2025) dataset tags each article by the type of data they share: participant-level summaries (_PARTICIPANT_), trial-level data (_TRIAL_), fixation-level data (_FIXATION_), or no data (_NONE_).<br>
Each article may share multiple types of data, but for our analysis we will focus on the _highest granularity_ of shared data, in the order of FIXATION > TRIAL > PARTICIPANT > NONE.<br>

In [ ]:
print(f"Final number of publications:\t {combined.shape[0]}")
print(f"Articles sharing any materials:\t{combined['is_sharing_anything'].sum()}")
print(f"Articles sharing data:\t{combined['is_sharing_data'].sum()}")

multi_sharing = (
    (combined[["BY_FIXATION", "BY_TRIAL", "BY_PPT"]] == "YES")
    .value_counts(dropna=False)
    .sort_index()
)
multi_sharing.index = multi_sharing.index.rename(
    {"BY_FIXATION": "FIXATION", "BY_TRIAL": "TRIAL", "BY_PPT": "PARTICIPANT"}
)

multi_sharing

### Publication Age
We measure publication age as the difference between the last time its OpenAlex metadata was updated and the publication date, in weeks. This value is precomputed and stored in our records as `Pub2UpdateTime`.<br>
Alternatively, we can look directly at the publication year, which is either stored as `YEAR_PUBLISHED` in the Godwin dataset or as `PublicationYear` in the OpenAlex metadata (with some discrepancies between the two, see above).

In [ ]:
publication_age_in_weeks = (combined["Pub2UpdateTime"] / pd.Timedelta(1, unit="W")).rename("AgeInWeeks")
print(f"Num records with missing age: {publication_age_in_weeks.isna().sum()} ({100 * publication_age_in_weeks.isna().sum() / FINAL_DATASET_SIZE :.1f}%)")
publication_age_in_weeks.describe().to_frame().T

In [ ]:
age_fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False, shared_yaxes=False,
    subplot_titles=["Publication Year", "Publication Age (Weeks)",],
    vertical_spacing=0.15,
)

age_fig.add_trace(
    row=1, col=1, trace=go.Histogram(
        name="Publication Year",
        x=combined["PublicationYear"],
        marker=dict(color="#ff7f0e"),
        xbins=dict(
            start=combined["PublicationYear"].min() - 0.5,
            end=combined["PublicationYear"].max() + 0.5,
            size=1
        )
))
age_fig.add_trace(
    row=2, col=1, trace=go.Histogram(
        name="Publication Age (Weeks)",
        x=publication_age_in_weeks,
        marker=dict(color="#1f77b4"),
        nbinsx=20,
))

# update layout and styling
age_fig.update_xaxes(
    row=1, col=1,
    # title=dict(text="year", font=AXIS_TITLE_FONT, standoff=2),
    tickmode='linear',
)
age_fig.update_xaxes(
    row=2, col=1,
    # title=dict(text="weeks", font=AXIS_TITLE_FONT, standoff=2),
    autorange="reversed",
)

age_fig.update_yaxes(title=dict(text="Count", font=AXIS_TITLE_FONT, standoff=2))

age_fig.update_layout(
    width=800, height=600,
    showlegend=False,
    margin=dict(t=60, b=40, l=40, r=20),
    template="simple_white",
)

if False:
    age_fig.show()

### Publication Counts per Author/Venue
We check how many different first- and last-authors there are in our dataset, and how many different articles each has published.<br>
Similarly, we count the number of unique publication venues (journals/conferences) and how many different articles were published in each.

In [ ]:
authors = combined["Authors"].dropna().map(lambda auths: auths.split("; "))
print(f"Articles with missing authorship: {combined["Authors"].isna().sum()} ({100 * combined["Authors"].isna().sum() / FINAL_DATASET_SIZE :.1f}%)")
num_missing_venue = combined["VenueName"].isna().sum() + (combined["VenueName"].map(len) < 1).sum()
print(f"Num records with missing/invalid venue: {num_missing_venue} ({100 * num_missing_venue / FINAL_DATASET_SIZE :.1f}%)")

print("\n### First Authors: ###")
first_author = authors.map(lambda auths: auths[0]).replace(
    # manually validated matches
    {"York, A": "York, AA"}
).rename("first author")
first_counts = first_author.value_counts().rename("FirstAuthor")
num_articles_with_repeated_firsts = first_counts[first_counts > 1].sum()
print(f"Unique first-authors: {len(first_counts)}")
print(f"Articles by first-authors with multiple publications in the dataset: {num_articles_with_repeated_firsts} ({100 * num_articles_with_repeated_firsts / FINAL_DATASET_SIZE :.1f}%)")

print("\n### Last Authors: ###")
last_author = authors.map(lambda auths: auths[-1]).replace(
    # manually validated matches
    {"Becker, S": "Becker, SI", "Hooge, I": "Hooge, ITC", "Jiang, YV" : "Jiang, YHV", "Zelisky, G" : "Zelinsky, GJ"}
).rename("last author")
last_counts = last_author.value_counts().rename("LastAuthor")
num_articles_with_repeated_lasts = last_counts[last_counts > 1].sum()
print(f"Unique last-authors: {len(last_counts)}")
print(f"Articles by last-author with multiple publications in the dataset: {num_articles_with_repeated_lasts} ({100 * num_articles_with_repeated_lasts / FINAL_DATASET_SIZE :.1f}%)")

print("\n### Venue: ###")
articles_per_venue = combined["VenueName"].value_counts().sort_values(ascending=False).rename("Venue")
num_articles_with_repeated_venue = articles_per_venue[articles_per_venue > 1].sum()
print(f"Unique venues:\t{len(articles_per_venue.index)}")
print(f"Articles in venues with multiple publications in the dataset: {num_articles_with_repeated_venue} ({100 * num_articles_with_repeated_venue / FINAL_DATASET_SIZE :.1f}%)")

qs = [0.25, 0.50, 0.75, 0.90, 0.95]
pd.concat([first_counts.describe(qs), last_counts.describe(qs), articles_per_venue.describe(qs)], axis=1).T

### Article-Count Distribution Figure
We plot the following distributions:
1. Number of articles per first-author
2. Number of articles per last-author
3. Number of articles per venue

In [ ]:
articles_count_distributions_fig = make_subplots(
    rows=2, cols=2, shared_xaxes=False, shared_yaxes=True,
    specs=[[{}, {}], [{"colspan": 2}, None]],
    subplot_titles=["First Authors", "Last Authors", "Venues"],
    vertical_spacing=0.3, horizontal_spacing=0.1,
)

# number of articles per first-author
articles_count_distributions_fig.add_trace(
    go.Histogram(
        name="First Authors",
        x=first_counts, xbins=dict(start=0.5, end=first_counts.max() + 0.5, size=1),
        marker=dict(color='#1b9e77'),
    ),
    row=1, col=1
)

# number of articles per last-author
articles_count_distributions_fig.add_trace(
    go.Histogram(
        name="Last Authors",
        x=last_counts, xbins=dict(start=0.5, end=last_counts.max() + 0.5, size=1),
        marker=dict(color='#d95f02'),
    ),
    row=1, col=2
)

# number of articles per venue
venue_counts = articles_per_venue.value_counts().sort_index()
articles_count_distributions_fig.add_trace(
    go.Bar(
        name="Venues",
        x=venue_counts.index, y=venue_counts,
        marker=dict(color='#7570b3'),
    ),
    row=2, col=1
)

# update layout and styling
articles_count_distributions_fig.update_xaxes(
    title=dict(text="Number of Articles Published", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
)
articles_count_distributions_fig.update_yaxes(
    title=dict(text="Number of Authors", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
    row=1, col=1
)
articles_count_distributions_fig.update_yaxes(
    title=dict(text="Number of Venues"),
    row=2, col=1
)

articles_count_distributions_fig.update_layout(
    width=700, height=500,
    title=dict(text="Number of Published Articles", font=TITLE_FONT),
    showlegend=False,
    margin=dict(t=60, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    articles_count_distributions_fig.show()

### Authors per Publication
We check the distribution of authour-counts in each publication.

In [ ]:
num_missing_auth_count = combined['NumAuthors'].isna().sum() + (combined['NumAuthors'] < 1).sum()
print(f"Number of articles with missing or invalid author counts: {num_missing_auth_count} ({100 * num_missing_auth_count / FINAL_DATASET_SIZE :.1f}%)")

combined["NumAuthors"].describe().to_frame().T

In [ ]:
auths_per_article = combined["NumAuthors"].value_counts().sort_index()

authors_per_article_fig = go.Figure()
authors_per_article_fig.add_trace(go.Bar(
    x=auths_per_article.index, y=auths_per_article
))

# update layout and styling
authors_per_article_fig.update_xaxes(
    title=dict(text="Number of Authors per Articles", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
    tickmode='linear', dtick=1
)
authors_per_article_fig.update_yaxes(
    title=dict(text="Number of Articles", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT
)

authors_per_article_fig.update_layout(
    width=600, height=300,
    title=dict(text="Number of Authors per Article", font=TITLE_FONT),
    showlegend=False,
    margin=dict(t=40, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    authors_per_article_fig.show()

### Impact Score Distribution
We check the distribution of the Impact-Factor heuristic we chose, which is either `Venue2yrMeanCitedness`, `VenueHIndex`, or `VenueI10Index`.

In [ ]:
impact = combined[VENUE_IMPACT_METRIC]
num_missing_impact = impact.isna().sum() + (impact < 0).sum()
print(f"Num records missing impact score: {num_missing_impact} ({100 * num_missing_impact / FINAL_DATASET_SIZE :.1f}%)")

qs = [0.25, 0.50, 0.75, 0.90, 0.95]
combined[VENUE_IMPACT_METRIC].describe(qs).to_frame().T

In [ ]:
impact_score_distribution_fig = go.Figure()
impact_score_distribution_fig.add_trace(go.Violin(
    x=impact, name="",
    side="positive", fillcolor='#7570b3', line_color='black',
    points="all", pointpos=0.5, marker=dict(color='#bc8ee0'),
))

# update layout and styling
impact_score_distribution_fig.update_xaxes(
    title=dict(text=VENUE_IMAPCT_METRIC_NAME, font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
    tickmode='linear', dtick=1
)
impact_score_distribution_fig.update_yaxes(
    title=dict(text=None, font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT
)

impact_score_distribution_fig.update_layout(
    width=600, height=300,
    title=dict(text="Impact Score Distribution", font=TITLE_FONT),
    showlegend=False,
    margin=dict(t=40, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    impact_score_distribution_fig.show()

### Venue & Impact Score

In [ ]:
# number of sharing/non-sharing articles per venue
venue_stats = (
    combined[["VenueName", "is_sharing_data"]]
    .dropna()
    .value_counts()
    .reset_index(drop=False)
)
venue_stats["share_label"] = venue_stats["is_sharing_data"].map({True: "SHARING", False: "NOT SHARING"})
venue_stats[VENUE_IMPACT_METRIC] = venue_stats["VenueName"].map(
    lambda venue: combined.loc[combined["VenueName"] == venue, VENUE_IMPACT_METRIC].iloc[0]
)
venue_stats["_total_articles"] = venue_stats["VenueName"].map(venue_stats.groupby("VenueName")["count"].sum())
venue_stats.sort_values(by=["_total_articles", VENUE_IMPACT_METRIC, "VenueName"], inplace=True, kind="stable", ascending=False)
venue_stats.drop(columns=["is_sharing_data", "_total_articles"], inplace=True)

layout = go.Layout(
    yaxis=dict(), xaxis=dict(), xaxis2=dict(overlaying='x', side='top', anchor='y')
)
venue_and_impact_fig = go.Figure(layout=layout)

# stacked bar trace of num sharing/non-sharing articles per venue
for sl in venue_stats["share_label"].unique():
    subset = venue_stats[venue_stats["share_label"] == sl]
    color = CLASS_COLORS[sl]
    venue_and_impact_fig.add_trace(go.Bar(
        name=f"{sl.title()} Data",
        legendgroup="Article Counts", showlegend=True,
        y=subset["VenueName"], x=subset["count"],
        orientation='h',
        marker=dict(color=color),
        xaxis='x'
    ))

# scatter plot of venue impact scores
venue_scores = venue_stats[["VenueName", VENUE_IMPACT_METRIC]].drop_duplicates()
venue_and_impact_fig.add_trace(go.Scatter(
    name=VENUE_IMAPCT_METRIC_NAME, legendgroup=VENUE_IMAPCT_METRIC_NAME, showlegend=True,
    y=venue_scores["VenueName"],
    x=venue_scores[VENUE_IMPACT_METRIC],
    mode='markers',
    marker=dict(color="#ff7f0e", size=8, symbol='diamond'),
    xaxis='x2'
))

# update layout and styling
venue_and_impact_fig.update_xaxes(
    selector=0,
    title=dict(text="Number of Articles", font=AXIS_TITLE_FONT, standoff=2),
    tickfont=AXIS_TICK_FONT
)
venue_and_impact_fig.update_xaxes(
    selector=1,
    title=dict(text=VENUE_IMAPCT_METRIC_NAME, font=AXIS_TITLE_FONT, standoff=2),
    tickfont=AXIS_TICK_FONT
)
venue_and_impact_fig.update_yaxes(
    title=dict(font=AXIS_TITLE_FONT, standoff=2),
    tickfont=AXIS_TICK_FONT,
    categoryorder="array", categoryarray=venue_stats["VenueName"].drop_duplicates().tolist()
)
venue_and_impact_fig.update_layout(
    width=1200, height=600,
    title=dict(
        text="Distribution of Publications and Journal Impact", font=TITLE_FONT,
        xref="container", xanchor="center", x=0.5,
        yref="container", yanchor="bottom", y=0.975
    ),
    barmode='stack',
    legend=dict(
        visible=False, orientation="h",
        yanchor="bottom", y=-0.23, yref="paper",
        xanchor="left", x=0.0, xref="paper",
        font=AXIS_TICK_FONT,
    ),
    margin=dict(t=75, b=5, l=250, r=5),
    template="simple_white",
)

if False:
    venue_and_impact_fig.show()

In [ ]:
# flip to True to export this figure into `output/`
if False:
    save_figure(venue_and_impact_fig, "venue_impact.png", width=1200, height=600)

### Has American Author + Is Open-Access + Has Pre-Print
We check the rate of these three binary features in our dataset.

In [ ]:
binaries = {'HasUSAuthor': 'Has US Author', 'IsOpenAccess': 'Is OA', 'HasPreprint': 'Has Pre-Print'}
for b in binaries:
    bin_name = binaries[b]
    print(f"### {bin_name} ###")
    num_missing_binary = combined[b].isna().sum()
    print(f"Num records missing `{bin_name}`: {num_missing_binary}")
    percent_binary = combined[combined[b]].shape[0] * 100 / FINAL_DATASET_SIZE
    print(f"% `{bin_name}`: {combined[b].sum()} artciles ({percent_binary:.1f}%)")
    print()

In [ ]:
binary_counts = pd.concat([combined[b].value_counts().rename(b) for b in binaries.keys()], axis=1)
binary_percents = (100 * binary_counts / binary_counts.sum(axis=0)).round(2).T

binary_percents_fig = go.Figure()
for b in binary_percents.columns:
    binary_percents_fig.add_trace(go.Bar(
        x=binary_percents.index,
        y=binary_percents[b],
        name=b, marker_color="green" if b else "red"
    ))

# update layout and styling
binary_percents_fig.update_xaxes(
    title=dict(text="Binary Feature", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT,
)
binary_percents_fig.update_yaxes(
    title=dict(text="% of Articles", font=AXIS_TITLE_FONT),
    tickfont=AXIS_TICK_FONT
)

binary_percents_fig.update_layout(
    width=600, height=400,
    title=dict(text="Binary Features", font=TITLE_FONT),
    barmode='stack',
    showlegend=False,
    margin=dict(t=40, b=0, l=0, r=0),
    template="simple_white", # Clean background for academic plots
)

if False:
    binary_percents_fig.show()

---
## Impact Scores by Sharing Category
A descriptive summary of the three impact measures we track - raw citation counts, the group-level h-index, and FWCI - broken down by sharing class and for the pooled sharing group.

In [ ]:
def calculate_h_index(citations: pd.Series) -> int:
    """Calculates h-index from a Series of citation counts."""
    if citations.empty:
        return 0
    # Sort descending (highest citations first)
    sorted_cits = sorted(citations.dropna().astype(int), reverse=True)
    # Count how many papers have citations >= their rank
    return sum(c >= i + 1 for i, c in enumerate(sorted_cits))

In [ ]:
_IMPACT_AGGS = {
    "TotalCitations": ["count", "median", "mean", "std", calculate_h_index],
    "FieldWeightedCitationIndex": ["count", "median", "mean", "std"],
}

metrics_per_sharing_class = (
    combined[["data_sharing_class", "TotalCitations", "FieldWeightedCitationIndex"]]
    .groupby("data_sharing_class")
    .agg(_IMPACT_AGGS)
)
metrics_for_any_data_sharing = (
    combined.loc[combined["is_sharing_data"], ["TotalCitations", "FieldWeightedCitationIndex"]]
    .agg(_IMPACT_AGGS)
)
display(metrics_per_sharing_class)
display(metrics_for_any_data_sharing)